<a href="https://colab.research.google.com/github/rawantamer140/maze-pathfinder-bfs/blob/main/maze_bfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import ipywidgets as W
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap
from collections import deque
from IPython.display import HTML, display, clear_output

plt.rcParams["animation.embed_limit"] = 60

# ----------------  the maze ----------------
maze = [
    [0,0,0,1,0,0,0,0],
    [1,1,0,1,0,1,1,0],
    [0,0,0,0,0,0,1,0],
    [0,1,1,1,1,0,1,0],
    [0,0,0,0,1,0,0,0],
    [1,1,1,0,1,1,1,0],
    [0,0,0,0,0,0,1,0],
    [0,1,1,1,1,0,0,0],
]
N = 8
MOVES = [(-1,0), (1,0), (0,-1), (0,1)]  # Up, Down, Left, Right

# ---------------- BFS ----------------
def bfs(start, goal):
    queue = deque([start])
    parent = {start: None}
    order = []
    while queue:
        cur = queue.popleft()
        order.append(cur)
        if cur == goal:
            break
        for dr, dc in MOVES:
            nxt = (cur[0] + dr, cur[1] + dc)
            if (0 <= nxt[0] < N and 0 <= nxt[1] < N
                    and maze[nxt[0]][nxt[1]] == 0 and nxt not in parent):
                parent[nxt] = cur
                queue.append(nxt)
    if goal not in parent:
        return None, order
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1], order

# ---------------- draw ----------------
# 0 ممر | 1 حيطة | 2 مستكشف | 3 مسار | 4 بداية | 5 هدف
CMAP = ListedColormap(["#f4f1de", "#2b2d42", "#8ecae6", "#ffb703", "#06d6a0", "#ef476f"])

def is_open(cell):
    return maze[cell[0]][cell[1]] == 0

def build_grid(start, goal, explored=(), path=()):
    grid = [row[:] for row in maze]
    for r, c in explored: grid[r][c] = 2
    for r, c in path:     grid[r][c] = 3
    if is_open(start): grid[start[0]][start[1]] = 4
    if is_open(goal):  grid[goal[0]][goal[1]] = 5
    return grid

def draw(ax, grid, start, goal, title):
    ax.clear()
    ax.imshow(grid, cmap=CMAP, vmin=0, vmax=5)
    for cell, label in ((start, "S"), (goal, "G")):
        if is_open(cell):
            ax.text(cell[1], cell[0], label, ha="center", va="center",
                    fontweight="bold", fontsize=13)
    ax.set_xticks([x - .5 for x in range(N + 1)])
    ax.set_yticks([y - .5 for y in range(N + 1)])
    ax.grid(color="white", linewidth=1.5)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_title(title, fontsize=11, fontweight="bold")

def make_animation(start, goal, path, order, interval):
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    total = len(order) + len(path or [])

    def frame(i):
        explored = order[:min(i + 1, len(order))]
        on_path = path[:i - len(order) + 1] if (path and i >= len(order)) else []
        if i < len(order) - 1 or (i == len(order) - 1 and path):
            phase = "Exploring..."
        elif path:
            phase = "Shortest path!"
        else:
            phase = "No path found"
        draw(ax, build_grid(start, goal, explored, on_path), start, goal,
             f"BFS - {phase}   (explored: {len(explored)})")

    anim = animation.FuncAnimation(fig, frame, frames=total, interval=interval)
    plt.close(fig)
    return anim.to_jshtml(default_mode="once")

def text_render(start, goal, path):
    on = set(path or [])
    lines = []
    for r in range(N):
        row = []
        for c in range(N):
            if (r, c) == start:      row.append("S")
            elif (r, c) == goal:     row.append("G")
            elif maze[r][c] == 1:    row.append("█")
            elif (r, c) in on:       row.append("*")
            else:                    row.append(".")
        lines.append(" ".join(row))
    return "\n".join(lines)

# ---------------- interface elements ----------------
def dd(label, val):
    return W.Dropdown(options=list(range(N)), value=val, description=label,
                      style={"description_width": "40px"},
                      layout=W.Layout(width="140px"))

s_r, s_c = dd("Row", 0), dd("Col", 0)
g_r, g_c = dd("Row", 7), dd("Col", 7)

speed = W.IntSlider(value=350, min=100, max=1000, step=50,
                    description="Speed", readout_format="d",
                    style={"description_width": "45px"},
                    layout=W.Layout(width="260px"))
speed_hint = W.HTML("<small style='color:#888'>milliseconds per frame (lower = faster)</small>")

solve_btn = W.Button(description="Solve with BFS", icon="play",
                     button_style="success",
                     layout=W.Layout(width="260px", height="40px"))
status = W.HTML()
preview_out = W.Output()
result_out = W.Output()

title = W.HTML(
    "<div style='padding:14px 18px;border-radius:12px;"
    "background:linear-gradient(90deg,#2b2d42,#4a4e8a);color:white'>"
    "<div style='font-size:22px;font-weight:bold'>🧩 Interactive AI Maze Path-Finder</div>"
    "<div style='opacity:.85'>Breadth-First Search · 8×8 grid · live animation</div></div>")

legend = W.HTML(
    "<div style='font-size:13px;line-height:1.9'>"
    "<span style='color:#06d6a0'>■</span> Start &nbsp; "
    "<span style='color:#ef476f'>■</span> Goal &nbsp; "
    "<span style='color:#8ecae6'>■</span> Explored &nbsp; "
    "<span style='color:#ffb703'>■</span> Path &nbsp; "
    "<span style='color:#2b2d42'>■</span> Wall</div>")

# ---------------- the logic ----------------
def get_cells():
    return (s_r.value, s_c.value), (g_r.value, g_c.value)

def validate(start, goal):
    if maze[start[0]][start[1]] == 1:
        return "⛔ START is a wall. Choose another cell."
    if maze[goal[0]][goal[1]] == 1:
        return "⛔ GOAL is a wall. Choose another cell."
    if start == goal:
        return "⛔ GOAL must be different from START."
    return None

def refresh(_=None):
    start, goal = get_cells()
    err = validate(start, goal)
    if err:
        status.value = f"<div style='color:#ef476f;font-weight:bold'>{err}</div>"
    else:
        status.value = "<div style='color:#06d6a0;font-weight:bold'>✅ Ready - press Solve</div>"
    solve_btn.disabled = err is not None
    with preview_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(3.8, 3.8))
        draw(ax, build_grid(start, goal), start, goal, "Preview")
        plt.show()

def card(label, value, color):
    return (f"<div style='flex:1;min-width:150px;padding:12px 16px;border-radius:12px;"
            f"background:{color};color:white;text-align:center'>"
            f"<div style='font-size:12px;opacity:.9'>{label}</div>"
            f"<div style='font-size:22px;font-weight:bold'>{value}</div></div>")

def on_solve(_):
    start, goal = get_cells()
    if validate(start, goal):
        return
    solve_btn.disabled = True
    solve_btn.description = "Building animation..."
    try:
        path, order = bfs(start, goal)
        with result_out:
            clear_output(wait=True)
            if path:
                cards = (card("Solution Status", "✅ Path found", "#2a9d8f") +
                         card("Solution Cost", f"{len(path) - 1} steps", "#4a4e8a") +
                         card("Search Cost", f"{len(order)} nodes", "#e07a5f"))
            else:
                cards = (card("Solution Status", "❌ No path", "#ef476f") +
                         card("Solution Cost", "-", "#4a4e8a") +
                         card("Search Cost", f"{len(order)} nodes", "#e07a5f"))
            display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin:12px 0'>{cards}</div>"))
            display(HTML("<pre style='font-size:16px;line-height:1.3'>"
                         + text_render(start, goal, path) + "</pre>"))
            display(HTML(make_animation(start, goal, path, order, speed.value)))
    finally:
        solve_btn.description = "Solve with BFS"
        refresh()

for w in (s_r, s_c, g_r, g_c):
    w.observe(refresh, names="value")
solve_btn.on_click(on_solve)

# --------------- interface assembly ----------------
start_box = W.VBox([W.HTML("<b style='color:#06d6a0'>🟢 START</b>"), s_r, s_c])
goal_box = W.VBox([W.HTML("<b style='color:#ef476f'>🔴 GOAL</b>"), g_r, g_c])
controls = W.VBox([W.HBox([start_box, goal_box]), speed, speed_hint,
                   solve_btn, status, legend])

display(W.VBox([title, W.HBox([controls, preview_out]), result_out]))
refresh()